# ERA5 to PRISM Downscaling (Georgia)
Tuned training workflow with extended ERA5 inputs, temporal-depth checks, and ablation analysis.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


## Data pipeline
- ERA5 monthly download with single-level and pressure-level variables
- PRISM daily raster targets
- Supervised samples with shape `[T, C, H, W] -> [1, H_high, W_high]`


In [ ]:
import json
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from datasets.prism_dataset import ERA5_PRISM_Dataset

ERA5_PATH = ROOT / "data_raw/era5_georgia_temp.nc"
PRISM_DIR = ROOT / "data_raw/prism"
EVAL_DIR = ROOT / "results/evaluation"
TRAIN_DIR = ROOT / "results/training_logs"
TUNE_DIR = ROOT / "results/tuning"
TEMPORAL_DIR = ROOT / "results/temporal_analysis"
ABLATION_DIR = ROOT / "results/ablation"
VIS_DIR = ROOT / "results/visualizations"

if ERA5_PATH.exists() and PRISM_DIR.exists():
    ds_era5 = xr.open_dataset(ERA5_PATH)
    print("ERA5 variables:", list(ds_era5.data_vars))
    ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=3, input_set="extended", verbose=False)
    x, y = ds[0]
    print("Extended input shape [T, C, H, W]:", tuple(x.shape))
    print("Target shape [C, H, W]:", tuple(y.shape))
else:
    print("Data missing. Run:")
    print("python data_pipeline/download_era5_georgia.py --year 2023 --month 1 --overwrite")
    print("python data_pipeline/download_prism.py --start-date 20230101 --days 30 --variable tmean")


## Training diagnostics

In [ ]:
cnn_curve = TRAIN_DIR / "cnn_loss_curve.png"
convlstm_curve = TRAIN_DIR / "convlstm_loss_curve.png"
cnn_log = TRAIN_DIR / "cnn_training_log.csv"
convlstm_log = TRAIN_DIR / "convlstm_training_log.csv"

if cnn_curve.exists():
    display(Markdown("### CNN loss curve"))
    display(Image(filename=str(cnn_curve)))
if convlstm_curve.exists():
    display(Markdown("### ConvLSTM loss curve"))
    display(Image(filename=str(convlstm_curve)))

if cnn_log.exists() and convlstm_log.exists():
    cdf = pd.read_csv(cnn_log)
    vdf = pd.read_csv(convlstm_log)
    summary = pd.DataFrame([
        {"model": "cnn", "best_val_loss": float(cdf["val_loss"].min())},
        {"model": "convlstm", "best_val_loss": float(vdf["val_loss"].min())},
    ])
    display(summary)
else:
    print("Run final training commands from README first.")


## Tuning results

In [ ]:
tune_csv = TUNE_DIR / "tuning_summary.csv"
best_json = TUNE_DIR / "best_config.json"

if tune_csv.exists():
    tune_df = pd.read_csv(tune_csv)
    tune_df = tune_df[tune_df["status"] == "ok"].sort_values("best_val_loss")
    display(Markdown("### Top sweep runs"))
    display(tune_df[["run_name", "model", "history_length", "learning_rate", "weight_decay", "best_val_loss"]].head(10))
if best_json.exists():
    display(Markdown("### Best config"))
    display(json.loads(best_json.read_text()))


## Temporal modeling validation

In [ ]:
temporal_csv = TEMPORAL_DIR / "temporal_summary.csv"
if temporal_csv.exists():
    temporal_df = pd.read_csv(temporal_csv)
    display(temporal_df)
    pivot = temporal_df.pivot(index="history_length", columns="model", values="rmse")
    display(Markdown("### RMSE by history length"))
    display(pivot)
else:
    print("Run: python training/run_temporal_analysis.py --input-set extended --histories 1 3 6")


## Input ablation

In [ ]:
ablation_csv = ABLATION_DIR / "ablation_summary.csv"
if ablation_csv.exists():
    ab_df = pd.read_csv(ablation_csv)
    display(ab_df)
else:
    print("Run: python training/run_ablation.py --model convlstm --input-sets t2m core4 extended --history-length 6")


## Final evaluation and visual diagnostics

In [ ]:
summary_csv = EVAL_DIR / "baselines_summary.csv"
comparison_img = EVAL_DIR / "model_comparison.png"
sample_img = VIS_DIR / "sample_prediction.png"
err_img = VIS_DIR / "error_map.png"

if summary_csv.exists():
    display(pd.read_csv(summary_csv))
if comparison_img.exists():
    display(Markdown("### Model comparison"))
    display(Image(filename=str(comparison_img)))
if sample_img.exists():
    display(Markdown("### Sample prediction"))
    display(Image(filename=str(sample_img)))
if err_img.exists():
    display(Markdown("### Error map"))
    display(Image(filename=str(err_img)))


## Key observations
- Weak early runs were mostly optimization and training-length issues.
- ConvLSTM benefits from temporal context and tuned optimization.
- Extended ERA5 variables improved ConvLSTM compared with reduced input sets.
- Current best run is ConvLSTM with extended inputs and tuned settings.
